# LangChain: Chains, Memory & RAG — Reference Notebook

> **Reference notebook**, using current LangChain 1.0 idioms (LCEL, `create_agent` +
> checkpointer, manual retriever composition) rather than the legacy `Chain`/`Memory` classes. See
> [`langchain.md`](./langchain.md) for the concepts and the legacy-API notes behind every choice made
> here, [`chroma.md`](./chroma.md) for the vector store, and [`README.md`](./README.md) for the fuller
> walkthrough this notebook distills.

**Methods covered:**
- Prompt templates (`ChatPromptTemplate`) and LCEL composition (`prompt | llm | StrOutputParser()`)
- Multi-step pipelines: single-value chaining and named-output chaining, both via LCEL
- Multi-turn conversational memory via `create_agent` + a checkpointer (`InMemorySaver`, thread-scoped)
- RAG: `WebBaseLoader` → `OpenAIEmbeddings` → `Chroma` → manual `retriever.invoke()` → LCEL generation
- A persona-driven conversational agent combining a system prompt with memory

**Use this as a reference when:** you need copy-paste-ready, current (LangChain 1.0) code for prompt
templates, chained LLM calls, conversational memory, or a manually-assembled RAG pipeline.

**Don't use this as a reference for:** the legacy `LLMChain`/`SequentialChain`/`ConversationChain`/
`RetrievalQA` APIs themselves — those are documented (for historical/course-fidelity reasons) in
[`langchain.md`](./langchain.md), not reproduced here.

In [ ]:
import textwrap

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

import warnings
warnings.filterwarnings("ignore")

In [ ]:
llm = ChatOpenAI(temperature=0.9)

In [ ]:
# A template's placeholders get filled per call -- the same template renders a different prompt
# for every new `cuisine` value, without touching the static wording around it.
name_prompt = ChatPromptTemplate.from_template(
    "Quero abrir um restaurante de comida {cuisine}. Sugira um nome chique para isso."
)

name_chain = name_prompt | llm | StrOutputParser()
print(name_chain.invoke({"cuisine": "Mexicana"}))

In [ ]:
items_prompt = ChatPromptTemplate.from_template("Sugira alguns itens do menu para {restaurant_name}.")
items_chain = items_prompt | llm | StrOutputParser()

# LCEL equivalent of SimpleSequentialChain: pipe the first chain's single string output directly
# into the next chain's single input.
simple_pipeline = (
    name_chain
    | (lambda restaurant_name: {"restaurant_name": restaurant_name})
    | items_chain
)
print(simple_pipeline.invoke({"cuisine": "Indiana"}))

In [ ]:
# LCEL equivalent of SequentialChain: RunnablePassthrough.assign keeps the first output
# (restaurant_name) around alongside the second (menu_items), instead of discarding it.
full_pipeline = {"restaurant_name": name_chain} | RunnablePassthrough.assign(
    menu_items=lambda x: items_chain.invoke(x)
)
result = full_pipeline.invoke({"cuisine": "Italiana"})
print(result)

In [ ]:
# Memory now lives on an agent's checkpointer, not on the chain itself -- thread_id scopes which
# conversation's history gets loaded on each call.
chat_llm = ChatOpenAI(temperature=0.7)
agent = create_agent(chat_llm, tools=[], checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "demo-conversation"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Que país venceu mais vezes a Copa do Mundo de Futebol?"}]},
    config,
)
print(response["messages"][-1].content)

In [ ]:
# Same thread_id -- the agent automatically has access to the previous turn via the checkpointer.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "E quem é o maior artilheiro da história das Copas?"}]},
    config,
)
print(response["messages"][-1].content)

In [ ]:
def print_response(text: str):
    print("\n".join(textwrap.wrap(text, width=100)))

In [ ]:
loader = WebBaseLoader(
    "https://blog.dsacademy.com.br/como-rag-retrieval-augmented-generation-funciona-para-personalizar-os-llms/"
)
documents = loader.load()

embeddings = OpenAIEmbeddings()
vector_store = Chroma.from_documents(documents, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

In [ ]:
rag_prompt = ChatPromptTemplate.from_template(
    """Você é um Engenheiro de IA de Nível Sênior.

{context}

Responda considerando as técnicas mais modernas que você conhecer.

Pergunta: {question}
Resposta:"""
)

rag_chain = rag_prompt | llm | StrOutputParser()

def answer(question):
    # Manual retrieval + composition -- the modern replacement for RetrievalQA.
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return rag_chain.invoke({"context": context, "question": question})

print_response(answer("Explique o que é RAG em 5 sentenças"))

In [ ]:
# create_agent's system_prompt fixes the persona; the checkpointer still handles history/input
# bookkeeping automatically -- no manual prompt template with history/input placeholders needed.
sales_agent = create_agent(
    ChatOpenAI(temperature=0),
    tools=[],
    system_prompt=(
        "Esta é uma conversa entre um cliente e um especialista em venda de carros esportivos. "
        "Você é o especialista em carros, conhece bem os modelos esportivos e deve sempre responder "
        "com a maior precisão possível."
    ),
    checkpointer=InMemorySaver(),
)
sales_config = {"configurable": {"thread_id": "sales-demo"}}

In [ ]:
# Scripted turns stand in for an interactive loop -- keeps the notebook runnable top-to-bottom
# without blocking on input().
customer_messages = [
    "Olá. Tudo bem?",
    "Qual o modelo mais novo de Ferrari?",
    "Qual o torque?",
]

for message in customer_messages:
    response = sales_agent.invoke({"messages": [{"role": "user", "content": message}]}, sales_config)
    print(f"Cliente: {message}")
    print_response("Especialista: " + response["messages"][-1].content)
    print()